In [ ]:
#| default_exp gcp

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore.all import *
import pulumi
import pulumi_gcp as gcp
from typing import Optional, List, Dict, Any
from pathlib import Path

## Storage

Google Cloud Storage bucket with security defaults

In [ ]:
#| export
@delegates(gcp.storage.Bucket.__init__)
def storage(name:str=None, # Bucket name
            location:str="US", # Bucket location
            versioning:bool=True, # Enable versioning
            encryption:bool=True, # Enable encryption
            public:bool=False, # Allow public access
            **kwargs):
    "Create GCS bucket with security defaults"
    bucket_name = name or f"pullup-{pulumi.get_project()}-{pulumi.get_stack()}"
    
    # Create bucket
    bucket = gcp.storage.Bucket(bucket_name,
        location=location,
        uniform_bucket_level_access=True,
        versioning=gcp.storage.BucketVersioningArgs(
            enabled=versioning) if versioning else None,
        public_access_prevention="enforced" if not public else "inherited",
        **kwargs)
    
    return bucket

## VPC Network

GCP VPC with subnets

In [ ]:
#| export
@delegates(gcp.compute.Network.__init__)
def network(name:str=None, # Network name
            auto_subnets:bool=False, # Auto create subnets
            subnets:List[Dict]=None, # Custom subnet configurations
            **kwargs):
    "Create GCP VPC network"
    network_name = name or f"pullup-{pulumi.get_project()}"
    
    # Create network
    net = gcp.compute.Network(network_name,
        auto_create_subnetworks=auto_subnets,
        **kwargs)
    
    created_subnets = []
    
    # Create custom subnets if provided
    if subnets and not auto_subnets:
        for i, subnet_config in enumerate(subnets):
            subnet = gcp.compute.Subnetwork(f"{network_name}-subnet-{i}",
                network=net.id,
                ip_cidr_range=subnet_config.get("cidr", f"10.0.{i}.0/24"),
                region=subnet_config.get("region", "us-central1"),
                private_ip_google_access=True)
            created_subnets.append(subnet)
    
    return dict(network=net, subnets=created_subnets)

## Cloud Run

GCP Cloud Run service deployment

In [ ]:
#| export
@delegates(gcp.cloudrun.Service.__init__)
def cloudrun(name:str=None, # Service name
             image:str="gcr.io/cloudrun/hello", # Container image
             location:str="us-central1", # Region
             port:int=8080, # Container port
             memory:str="256Mi", # Memory limit
             cpu:str="1", # CPU limit
             public:bool=True, # Allow unauthenticated access
             **kwargs):
    "Create GCP Cloud Run service"
    service_name = name or f"pullup-{pulumi.get_project()}"
    
    # Create Cloud Run service
    service = gcp.cloudrun.Service(service_name,
        location=location,
        template=gcp.cloudrun.ServiceTemplateArgs(
            spec=gcp.cloudrun.ServiceTemplateSpecArgs(
                containers=[gcp.cloudrun.ServiceTemplateSpecContainerArgs(
                    image=image,
                    ports=[gcp.cloudrun.ServiceTemplateSpecContainerPortArgs(
                        container_port=port)],
                    resources=gcp.cloudrun.ServiceTemplateSpecContainerResourcesArgs(
                        limits={
                            "memory": memory,
                            "cpu": cpu
                        })
                )]
            )
        ),
        **kwargs)
    
    # Allow unauthenticated access if public
    if public:
        gcp.cloudrun.IamMember(f"{service_name}-public",
            service=service.name,
            location=location,
            role="roles/run.invoker",
            member="allUsers")
    
    return service

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()